# Week 1 Lab: Supplement Sales KPIs

## KPI 1: Total Revenue per Country

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
import plotly.graph_objects as go


warnings.filterwarnings("ignore")

# Economist-style chart template (used by every chart below)
economist = go.layout.Template()

economist.layout = go.Layout(
    font=dict(family="Helvetica Neue, Helvetica, Arial, sans-serif", size=13, color="#0C0C0C"),
    title=dict(x=0, xref="paper", xanchor="left", y=0.94, yref="container", yanchor="top", font=dict(size=20)),
    colorway=["#006BA2", "#3EBCD2", "#B28A00", "#7A3E6E", "#379A8B",
              "#EBB434", "#9A607F", "#758D99", "#D1B07C", "#B4BA39"],
    paper_bgcolor="white",
    plot_bgcolor="white",
    height=520,
    margin=dict(l=40, r=70, t=120, b=70),
    xaxis=dict(showgrid=False, showline=True, linecolor="#0C0C0C", linewidth=1,
               ticks="outside", tickcolor="#0C0C0C", ticklen=5,
               dtick="M12", tickformat="%Y", hoverformat="%b %Y"),
    yaxis=dict(side="right", showgrid=True, gridcolor="#B7C6CF", gridwidth=1,
               showline=False, ticks="", zeroline=True, zerolinecolor="#0C0C0C", zerolinewidth=1.5),
    legend=dict(orientation="h", x=0, xanchor="left", y=1.02, yanchor="bottom", title=dict(text="")),
    hoverlabel=dict(bgcolor="white", font_size=12),
    shapes=[
        # red rule along the top and the red tag at the top left
        dict(name="red rule", type="rect", xref="paper", yref="paper", x0=0, x1=1, y0=1.3576, y1=1.3636,
             fillcolor="#E3120B", line_width=0),
        dict(name="red tag", type="rect", xref="paper", yref="paper", x0=0, x1=0.04, y0=1.3212, y1=1.3636,
             fillcolor="#E3120B", line_width=0)
    ],
    annotations=[
        dict(name="source", text="Source: Supplement Sales Weekly data", xref="paper", yref="paper",
             x=0, y=-0.13, xanchor="left", yanchor="top", showarrow=False,
             font=dict(size=11, color="#758D99"))
    ]
)

economist.data.scatter = [go.Scatter(line=dict(width=2.5), marker=dict(size=5))]

df = pd.read_csv("Supplement_Sales_Weekly_Expanded.csv")

df["Date"] = pd.to_datetime(df["Date"])

df["Monthly"] = df["Date"].dt.to_period("M")
monthly = df.groupby("Monthly").agg({
    "Revenue" : "sum",
    "Units Sold" : "sum",
    "Discount" : "mean",
    "Units Returned" : "sum"
}).reset_index()  

monthly["Monthly"] = monthly["Monthly"].astype(str)

monthly_country = (
    df.groupby(['Location', 'Monthly'])['Revenue']
      .sum()
      .reset_index()
)

# Pivot so each country becomes a separate column
pivot = monthly_country.pivot(index='Monthly', columns='Location', values='Revenue')

# Sort chronologically
pivot = pivot.sort_index(ascending=True)

# Convert PeriodIndex to string for plotting
pivot.index = pivot.index.astype(str)

# Calculate % Change
pct_change = pivot.pct_change() * 100
pct_change.index = pct_change.index.astype(str)

fig = go.Figure()

for country in pct_change.columns:
    series = pct_change[country]

    # Main line trace
    fig.add_trace(go.Scatter(
        x=series.index,
        y=series.values,
        mode='lines+markers',
        name=country,
        legendgroup=country,        # link group
        hovertemplate=f"{country}: %{{y:.2f}}%"
    ))

    # Latest point
    latest_idx = series.last_valid_index()
    latest_val = series.loc[latest_idx]

    fig.add_trace(go.Scatter(
        x=[latest_idx],
        y=[latest_val],
        mode='markers+text',
        marker=dict(color='red', size=12),
        text=[f"{latest_val:.2f}%"],
        textposition="top center",
        name=f"{country} latest",
        legendgroup=country,        # link to same group
        showlegend=False,           # keeps legend clean
        hoverinfo="skip"
    ))

fig.update_layout(
    title="<b>Month-Over-Month Revenue % Change by Country</b><br><span style='font-size:14px'>% change on previous month</span>",
    hovermode="x unified",
    template=economist
)

fig.show()


In [2]:

# Create interactive figure
fig = go.Figure()

for country in pivot.columns:
    series = pivot[country]

    # Main line trace
    fig.add_trace(go.Scatter(
        x=series.index,
        y=series.values,
        mode='lines+markers',
        name=country,
        legendgroup=country,        # link group
        hovertemplate=f"{country}<br>Revenue: $%{{y:,.2f}}"
    ))

    # Find most recent non-NaN value
    latest_idx = series.last_valid_index()
    latest_val = series.loc[latest_idx]

    # Highlight marker
    fig.add_trace(go.Scatter(
        x=[latest_idx],
        y=[latest_val],
        mode='markers+text',
        marker=dict(color='red', size=12),
        text=[f"${latest_val:,.0f}"],
        textposition="top center",
        name=f"{country} latest",
        legendgroup=country,        # linked to same group
        showlegend=False,           # keeps legend clean
        hoverinfo="skip"
    ))

fig.update_layout(
    title="<b>Total Revenue per Month by Country</b><br><span style='font-size:14px'>Monthly revenue, $</span>",
    hovermode="x unified",
    template=economist
)

fig.show()

In [3]:

# Build monthly totals per country
pivot = monthly_country.pivot(index='Monthly', columns='Location', values='Revenue')
pivot = pivot.sort_index(ascending=True)

# Compute % change
pct_change = pivot.pct_change() * 100

# Identify latest month
latest_month = pivot.index[-1]

# Build table with countries as rows
rows = []

for country in pivot.columns:
    latest_revenue = pivot.loc[latest_month, country]
    latest_pct = pct_change.loc[latest_month, country]

    rows.append({
        "Country": country,
        "Latest Month": str(latest_month),
        "Total Revenue": latest_revenue,
        "% Change MoM": latest_pct
    })

summary_table = pd.DataFrame(rows)

# Conditional formatting function
def color_pct(val):
    if pd.isna(val):
        return ""
    return "color: green;" if val > 0 else "color: red;"

# Apply formatting
styled_table = (
    summary_table.style
        .hide(axis="index")
        
        .format({
            "Total Revenue": "${:,.2f}",
            "% Change MoM": "{:.2f}%"
        })
        .map(color_pct, subset=["% Change MoM"])
)

styled_table

Country,Latest Month,Total Revenue,% Change MoM
Canada,2025-03,"$114,970.03",5.49%
UK,2025-03,"$142,951.72",1.40%
USA,2025-03,"$117,937.26",28.47%


## KPI 2: Platform Revenue per Country

In [4]:
# Build monthly revenue per country per platform
monthly_platform = (
    df.groupby(['Location', 'Platform', 'Monthly'])['Revenue']
      .sum()
      .reset_index()
)

# Pivot so each Country / Platform pair becomes a separate column
pivot_platform = monthly_platform.pivot(index='Monthly', columns=['Location', 'Platform'], values='Revenue')

# Sort chronologically
pivot_platform = pivot_platform.sort_index(ascending=True)

# Months with no sales count as 0 revenue
pivot_platform = pivot_platform.fillna(0)

# Compute % change
pct_platform = pivot_platform.pct_change() * 100
pct_platform = pct_platform.replace([np.inf, -np.inf], np.nan)   # change from a $0 month is undefined

# Identify latest month
latest_month = pivot_platform.index[-1]

# Build table with Country / Platform as rows
rows = []

for country, platform in pivot_platform.columns:
    latest_revenue = pivot_platform.loc[latest_month, (country, platform)]
    latest_pct = pct_platform.loc[latest_month, (country, platform)]

    rows.append({
        "Country": country,
        "Platform": platform,
        "Latest Month": str(latest_month),
        "Total Revenue": latest_revenue,
        "% Change MoM": latest_pct
    })

platform_table = pd.DataFrame(rows)

# Apply formatting
styled_platform = (
    platform_table.style
        .hide(axis="index")
        .format({
            "Total Revenue": "${:,.2f}",
            "% Change MoM": "{:.2f}%"
        }, na_rep="n/a")
        .map(color_pct, subset=["% Change MoM"])
)

styled_platform


Country,Platform,Latest Month,Total Revenue,% Change MoM
Canada,Amazon,2025-03,"$42,524.01",20.99%
Canada,Walmart,2025-03,"$44,202.94",71.19%
Canada,iHerb,2025-03,"$28,243.08",-41.18%
UK,Amazon,2025-03,"$51,813.94",18.04%
UK,Walmart,2025-03,"$37,128.06",-15.92%
UK,iHerb,2025-03,"$54,009.72",2.03%
USA,Amazon,2025-03,"$31,770.52",-8.25%
USA,Walmart,2025-03,"$46,408.16",106.60%
USA,iHerb,2025-03,"$39,758.58",14.55%


In [5]:
# Monthly revenue trend: one chart per country, one line per platform
plot_platform = pivot_platform.copy()
plot_platform.index = plot_platform.index.astype(str)

for country in plot_platform.columns.get_level_values('Location').unique():

    fig = go.Figure()

    for platform in plot_platform[country].columns:
        series = plot_platform[country][platform]

        # Main line trace
        fig.add_trace(go.Scatter(
            x=series.index,
            y=series.values,
            mode='lines+markers',
            name=platform,
            legendgroup=platform,
            hovertemplate=f"{platform}<br>Revenue: $%{{y:,.2f}}"
        ))

        # Latest point
        latest_idx = series.last_valid_index()
        latest_val = series.loc[latest_idx]

        fig.add_trace(go.Scatter(
            x=[latest_idx],
            y=[latest_val],
            mode='markers+text',
            marker=dict(color='red', size=12),
            text=[f"${latest_val:,.0f}"],
            textposition="top center",
            name=f"{platform} latest",
            legendgroup=platform,
            showlegend=False,
            hoverinfo="skip"
        ))

    fig.update_layout(
        title=f"<b>{country}: Revenue per Month by Platform</b><br><span style='font-size:14px'>Monthly revenue, $</span>",
        hovermode="x unified",
        template=economist
    )

    fig.show()


In [6]:
# Month-over-month % change: one chart per country, one line per platform
plot_pct_platform = pct_platform.copy()
plot_pct_platform.index = plot_pct_platform.index.astype(str)

for country in plot_pct_platform.columns.get_level_values('Location').unique():

    fig = go.Figure()

    for platform in plot_pct_platform[country].columns:
        series = plot_pct_platform[country][platform]

        # Main line trace
        fig.add_trace(go.Scatter(
            x=series.index,
            y=series.values,
            mode='lines+markers',
            name=platform,
            legendgroup=platform,
            hovertemplate=f"{platform}: %{{y:.2f}}%"
        ))

        # Latest point
        latest_idx = series.last_valid_index()
        latest_val = series.loc[latest_idx]

        fig.add_trace(go.Scatter(
            x=[latest_idx],
            y=[latest_val],
            mode='markers+text',
            marker=dict(color='red', size=12),
            text=[f"{latest_val:.2f}%"],
            textposition="top center",
            name=f"{platform} latest",
            legendgroup=platform,
            showlegend=False,
            hoverinfo="skip"
        ))

    fig.update_layout(
        title=f"<b>{country}: Month-Over-Month Revenue % Change by Platform</b><br><span style='font-size:14px'>% change on previous month</span>",
        hovermode="x unified",
        template=economist
    )

    fig.show()


## KPI 3: Platform Revenue per Country per Category

Not every category sells every month on every platform, so months with no sales are filled with 0. A % change from a $0 month is undefined and shows as blank / n/a.

In [7]:
# Build monthly revenue per country per platform per category
monthly_category = (
    df.groupby(['Location', 'Platform', 'Category', 'Monthly'])['Revenue']
      .sum()
      .reset_index()
)

# Pivot so each Country / Platform / Category combination becomes a separate column
pivot_category = monthly_category.pivot(index='Monthly', columns=['Location', 'Platform', 'Category'], values='Revenue')

# Sort chronologically
pivot_category = pivot_category.sort_index(ascending=True)

# Months with no sales count as 0
pivot_category = pivot_category.fillna(0)

# Compute % change
pct_category = pivot_category.pct_change() * 100
pct_category = pct_category.replace([np.inf, -np.inf], np.nan)   # change from a $0 month is undefined

# Identify latest month
latest_month = pivot_category.index[-1]

# Build table with Country / Platform / Category as rows
rows = []

for country, platform, category in pivot_category.columns:
    latest_value = pivot_category.loc[latest_month, (country, platform, category)]
    latest_pct = pct_category.loc[latest_month, (country, platform, category)]

    rows.append({
        "Country": country,
        "Platform": platform,
        "Category": category,
        "Latest Month": str(latest_month),
        "Total Revenue": latest_value,
        "% Change MoM": latest_pct
    })

category_table = pd.DataFrame(rows)

# Apply formatting
styled_category = (
    category_table.style
        .hide(axis="index")
        .format({
            "Total Revenue": "${:,.2f}",
            "% Change MoM": "{:.2f}%"
        }, na_rep="n/a")
        .map(color_pct, subset=["% Change MoM"])
)

styled_category


Country,Platform,Category,Latest Month,Total Revenue,% Change MoM
Canada,Amazon,Amino Acid,2025-03,"$1,784.64",n/a
Canada,Amazon,Fat Burner,2025-03,$0.00,n/a
Canada,Amazon,Herbal,2025-03,"$8,421.60",n/a
Canada,Amazon,Hydration,2025-03,"$4,440.84",-47.17%
Canada,Amazon,Mineral,2025-03,"$13,051.87",141.27%
Canada,Amazon,Omega,2025-03,$0.00,-100.00%
Canada,Amazon,Performance,2025-03,"$9,525.84",150.13%
Canada,Amazon,Protein,2025-03,$0.00,-100.00%
Canada,Amazon,Sleep Aid,2025-03,$0.00,-100.00%
Canada,Amazon,Vitamin,2025-03,"$5,299.22",n/a


In [8]:
# Monthly revenue trend: one chart per country / platform, one line per category
plot_category = pivot_category.copy()
plot_category.index = plot_category.index.astype(str)

for country in plot_category.columns.get_level_values('Location').unique():
    for platform in plot_category[country].columns.get_level_values('Platform').unique():

        fig = go.Figure()

        for category in plot_category[country][platform].columns:
            series = plot_category[country][platform][category]

            # Main line trace
            fig.add_trace(go.Scatter(
                x=series.index,
                y=series.values,
                mode='lines+markers',
                name=category,
                legendgroup=category,
                hovertemplate=f"{category}<br>Revenue: $%{{y:,.2f}}"
            ))

            # Latest point
            latest_idx = series.last_valid_index()
            if latest_idx is None:
                continue
            latest_val = series.loc[latest_idx]

            fig.add_trace(go.Scatter(
                x=[latest_idx],
                y=[latest_val],
                mode='markers+text',
                marker=dict(color='red', size=12),
                text=[f"${latest_val:,.0f}"],
                textposition="top center",
                name=f"{category} latest",
                legendgroup=category,
                showlegend=False,
                hoverinfo="skip"
            ))

        fig.update_layout(
            title=f"<b>{country} / {platform}: Revenue per Month by Category</b><br><span style='font-size:14px'>Monthly revenue, $</span>",
            hovermode="x unified",
            template=economist
        )

        fig.show()


In [9]:
# Month-over-month % change: one chart per country / platform, one line per category
plot_pct_category = pct_category.copy()
plot_pct_category.index = plot_pct_category.index.astype(str)

for country in plot_pct_category.columns.get_level_values('Location').unique():
    for platform in plot_pct_category[country].columns.get_level_values('Platform').unique():

        fig = go.Figure()

        for category in plot_pct_category[country][platform].columns:
            series = plot_pct_category[country][platform][category]

            # Main line trace
            fig.add_trace(go.Scatter(
                x=series.index,
                y=series.values,
                mode='lines+markers',
                name=category,
                legendgroup=category,
                hovertemplate=f"{category}: %{{y:.2f}}%"
            ))

            # Latest point
            latest_idx = series.last_valid_index()
            if latest_idx is None:
                continue
            latest_val = series.loc[latest_idx]

            fig.add_trace(go.Scatter(
                x=[latest_idx],
                y=[latest_val],
                mode='markers+text',
                marker=dict(color='red', size=12),
                text=[f"{latest_val:.2f}%"],
                textposition="top center",
                name=f"{category} latest",
                legendgroup=category,
                showlegend=False,
                hoverinfo="skip"
            ))

        fig.update_layout(
            title=f"<b>{country} / {platform}: Month-Over-Month Revenue % Change by Category</b><br><span style='font-size:14px'>% change on previous month</span>",
            hovermode="x unified",
            template=economist
        )

        fig.show()


## KPI 4: Platform Returns per Country per Category

Same layout as KPI 3, using `Units Returned` instead of `Revenue`.

In [10]:
# Build monthly returns per country per platform per category
monthly_returns = (
    df.groupby(['Location', 'Platform', 'Category', 'Monthly'])['Units Returned']
      .sum()
      .reset_index()
)

# Pivot so each Country / Platform / Category combination becomes a separate column
pivot_returns = monthly_returns.pivot(index='Monthly', columns=['Location', 'Platform', 'Category'], values='Units Returned')

# Sort chronologically
pivot_returns = pivot_returns.sort_index(ascending=True)

# Months with no sales count as 0
pivot_returns = pivot_returns.fillna(0)

# Compute % change
pct_returns = pivot_returns.pct_change() * 100
pct_returns = pct_returns.replace([np.inf, -np.inf], np.nan)   # change from a 0-return month is undefined

# Identify latest month
latest_month = pivot_returns.index[-1]

# Build table with Country / Platform / Category as rows
rows = []

for country, platform, category in pivot_returns.columns:
    latest_value = pivot_returns.loc[latest_month, (country, platform, category)]
    latest_pct = pct_returns.loc[latest_month, (country, platform, category)]

    rows.append({
        "Country": country,
        "Platform": platform,
        "Category": category,
        "Latest Month": str(latest_month),
        "Total Returns": latest_value,
        "% Change MoM": latest_pct
    })

returns_table = pd.DataFrame(rows)

# Apply formatting
styled_returns = (
    returns_table.style
        .hide(axis="index")
        .format({
            "Total Returns": "{:,.0f}",
            "% Change MoM": "{:.2f}%"
        }, na_rep="n/a")
        .map(color_pct, subset=["% Change MoM"])
)

styled_returns


Country,Platform,Category,Latest Month,Total Returns,% Change MoM
Canada,Amazon,Amino Acid,2025-03,0,n/a
Canada,Amazon,Fat Burner,2025-03,0,n/a
Canada,Amazon,Herbal,2025-03,2,n/a
Canada,Amazon,Hydration,2025-03,1,-80.00%
Canada,Amazon,Mineral,2025-03,7,600.00%
Canada,Amazon,Omega,2025-03,0,n/a
Canada,Amazon,Performance,2025-03,2,0.00%
Canada,Amazon,Protein,2025-03,0,-100.00%
Canada,Amazon,Sleep Aid,2025-03,0,n/a
Canada,Amazon,Vitamin,2025-03,3,n/a


In [11]:
# Monthly returns trend: one chart per country / platform, one line per category
plot_returns = pivot_returns.copy()
plot_returns.index = plot_returns.index.astype(str)

for country in plot_returns.columns.get_level_values('Location').unique():
    for platform in plot_returns[country].columns.get_level_values('Platform').unique():

        fig = go.Figure()

        for category in plot_returns[country][platform].columns:
            series = plot_returns[country][platform][category]

            # Main line trace
            fig.add_trace(go.Scatter(
                x=series.index,
                y=series.values,
                mode='lines+markers',
                name=category,
                legendgroup=category,
                hovertemplate=f"{category}<br>Returns: %{{y:,.0f}}"
            ))

            # Latest point
            latest_idx = series.last_valid_index()
            if latest_idx is None:
                continue
            latest_val = series.loc[latest_idx]

            fig.add_trace(go.Scatter(
                x=[latest_idx],
                y=[latest_val],
                mode='markers+text',
                marker=dict(color='red', size=12),
                text=[f"{latest_val:,.0f}"],
                textposition="top center",
                name=f"{category} latest",
                legendgroup=category,
                showlegend=False,
                hoverinfo="skip"
            ))

        fig.update_layout(
            title=f"<b>{country} / {platform}: Returns per Month by Category</b><br><span style='font-size:14px'>Monthly units returned</span>",
            hovermode="x unified",
            template=economist
        )

        fig.show()


In [12]:
# Month-over-month % change: one chart per country / platform, one line per category
plot_pct_returns = pct_returns.copy()
plot_pct_returns.index = plot_pct_returns.index.astype(str)

for country in plot_pct_returns.columns.get_level_values('Location').unique():
    for platform in plot_pct_returns[country].columns.get_level_values('Platform').unique():

        fig = go.Figure()

        for category in plot_pct_returns[country][platform].columns:
            series = plot_pct_returns[country][platform][category]

            # Main line trace
            fig.add_trace(go.Scatter(
                x=series.index,
                y=series.values,
                mode='lines+markers',
                name=category,
                legendgroup=category,
                hovertemplate=f"{category}: %{{y:.2f}}%"
            ))

            # Latest point
            latest_idx = series.last_valid_index()
            if latest_idx is None:
                continue
            latest_val = series.loc[latest_idx]

            fig.add_trace(go.Scatter(
                x=[latest_idx],
                y=[latest_val],
                mode='markers+text',
                marker=dict(color='red', size=12),
                text=[f"{latest_val:.2f}%"],
                textposition="top center",
                name=f"{category} latest",
                legendgroup=category,
                showlegend=False,
                hoverinfo="skip"
            ))

        fig.update_layout(
            title=f"<b>{country} / {platform}: Month-Over-Month Returns % Change by Category</b><br><span style='font-size:14px'>% change on previous month</span>",
            hovermode="x unified",
            template=economist
        )

        fig.show()


## Product Rankings: Latest Month Top 5 per Country

In [13]:
# Filter to the latest month
latest_month = df['Monthly'].max()
latest_df = df[df['Monthly'] == latest_month]

# Top 5 products by revenue per country
top_revenue = (
    latest_df.groupby(['Location', 'Product Name'])['Revenue']
      .sum()
      .reset_index()
      .sort_values(['Location', 'Revenue', 'Product Name'], ascending=[True, False, True])
      .groupby('Location')
      .head(5)
)

top_revenue["Rank"] = top_revenue.groupby('Location').cumcount() + 1
top_revenue = top_revenue.rename(columns={"Location": "Country", "Product Name": "Product"})
top_revenue = top_revenue[["Country", "Rank", "Product", "Revenue"]]

styled_top_revenue = (
    top_revenue.style
        .hide(axis="index")
        .set_caption(f"Top 5 Products by Revenue per Country ({latest_month})")
        .format({"Revenue": "${:,.2f}"})
)

styled_top_revenue


Country,Rank,Product,Revenue
Canada,1,Vitamin C,"$18,943.62"
Canada,2,Zinc,"$17,334.72"
Canada,3,Creatine,"$13,135.38"
Canada,4,Collagen Peptides,"$11,737.15"
Canada,5,Ashwagandha,"$8,421.60"
UK,1,Biotin,"$20,404.80"
UK,2,Iron Supplement,"$17,352.96"
UK,3,Magnesium,"$14,786.41"
UK,4,Pre-Workout,"$14,675.49"
UK,5,Vitamin C,"$11,584.10"


In [14]:
# Top 5 products by returns per country
top_returns = (
    latest_df.groupby(['Location', 'Product Name'])['Units Returned']
      .sum()
      .reset_index()
      .sort_values(['Location', 'Units Returned', 'Product Name'], ascending=[True, False, True])
      .groupby('Location')
      .head(5)
)

top_returns["Rank"] = top_returns.groupby('Location').cumcount() + 1
top_returns = top_returns.rename(columns={"Location": "Country", "Product Name": "Product"})
top_returns = top_returns[["Country", "Rank", "Product", "Units Returned"]]

styled_top_returns = (
    top_returns.style
        .hide(axis="index")
        .set_caption(f"Top 5 Products by Units Returned per Country ({latest_month})")
        .format({"Units Returned": "{:,.0f}"})
)

styled_top_returns


Country,Rank,Product,Units Returned
Canada,1,Vitamin C,12
Canada,2,Zinc,7
Canada,3,Whey Protein,6
Canada,4,Collagen Peptides,4
Canada,5,Magnesium,3
UK,1,Biotin,6
UK,2,Melatonin,6
UK,3,Ashwagandha,5
UK,4,Iron Supplement,5
UK,5,Magnesium,5
